# Mecha-nudges for Machines — Walkthrough

This notebook walks through the key ideas and code behind the paper [Mecha-nudges for Machines](https://arxiv.org/abs/2603.23433). We cover:

1. **The measurement problem**: How do you quantify whether text is optimized for AI agents?
2. **Constrained generation**: Forcing an LLM to make a binary decision with calibrated probabilities
3. **Fine-tuning**: Training the classifiers used to compute $\mathcal{F}$-Information
4. **$\mathcal{F}$-Information**: Computing how much product text reduces an LLM's decision uncertainty
5. **Main result**: Reproducing the paper's regression on the included data

## 1. The Measurement Problem

After ChatGPT's release (November 30, 2022), sellers on platforms like Etsy may have started optimizing their product listings for AI agents — not just humans. But how do you measure this?

We use **Pointwise $\mathcal{F}$-Information (PVI)** from [Xu et al. (2020)](https://arxiv.org/abs/2003.12298):

$$\text{PVI}(x) = H_{\mathcal{F}}(Y \mid \emptyset) - H_{\mathcal{F}}(Y \mid x)$$

- $H_{\mathcal{F}}(Y \mid \emptyset)$: How uncertain is the model when it sees **no text**? (baseline)
- $H_{\mathcal{F}}(Y \mid x)$: How uncertain is the model when it sees **the actual product text**?
- The difference tells us how many **bits** of useful information the text provides to the AI.

If post-ChatGPT listings have higher PVI than pre-ChatGPT listings, the text has become more *machine-readable* — consistent with mecha-nudging.

## 2. Constrained Generation (Label Creation)

The first pipeline step prompts an LLM to make a binary SELECT/PASS decision for each product listing. Rather than parsing free-text, we use a custom `LogitsProcessor` that constrains output to exactly one token.

### The prompt

We frame the LLM as a selective Etsy browsing assistant (prompt V4):

In [ ]:
with open("prompts/prompt_v4.txt") as f:
    prompt_template = f.read()

print(prompt_template)

The prompt uses three placeholders: `{product_info}`, `{positive_token}`, `{negative_token}`. Here's how a prompt is constructed for a specific product:

In [ ]:
import sys
sys.path.insert(0, "src")
from generation_utils import prompt_creator

# Example product
product = {
    "Title": "Handmade Ceramic Mug - Speckled Blue Glaze",
    "Item_Details": "Wheel-thrown stoneware mug with a unique speckled blue glaze. Holds 12oz. Dishwasher and microwave safe."
}

full_prompt, product_info = prompt_creator(product, positive_token="SELECT", negative_token="PASS")
print(full_prompt)

### The constrained decoding mechanism

The key innovation is `PosNegLogitsProcessor`. On each generation step:

1. **Step 1** (first token): Mask the entire vocabulary except the two decision tokens. Extract calibrated probabilities via softmax over those two logits. Output the higher-probability token.
2. **Step 2+**: Force the EOS token to stop generation.

This guarantees every response is a valid binary decision, and gives us calibrated P(SELECT) and P(PASS).

In [ ]:
import torch
from generation_utils import PosNegLogitsProcessor

# Core logic of the constrained decoding (simplified for illustration)
# In practice this runs inside HuggingFace's generate() loop

def explain_constrained_decoding():
    """Walk through the logic of PosNegLogitsProcessor."""
    
    # Suppose the model produces these raw logits over its vocabulary
    vocab_size = 32000
    raw_logits = torch.randn(1, vocab_size)  # (batch=1, vocab)
    
    # Token IDs for SELECT and PASS (hypothetical)
    select_id, pass_id = 5765, 15114
    
    # Step 1: Extract logits for just the two decision tokens
    decision_logits = raw_logits[:, [select_id, pass_id]]
    print(f"Raw logits — SELECT: {decision_logits[0, 0]:.3f}, PASS: {decision_logits[0, 1]:.3f}")
    
    # Calibrated probabilities via softmax
    probs = torch.nn.functional.softmax(decision_logits, dim=-1)
    print(f"P(SELECT) = {probs[0, 0]:.4f}")
    print(f"P(PASS)   = {probs[0, 1]:.4f}")
    
    # Mask everything else to -inf, output the winner
    masked = torch.full_like(raw_logits, -float("inf"))
    if probs[0, 0] > probs[0, 1]:
        masked[0, select_id] = 1e4
        print("→ Decision: SELECT")
    else:
        masked[0, pass_id] = 1e4
        print("→ Decision: PASS")

explain_constrained_decoding()

## 3. Fine-Tuning

After labeling all listings with SELECT/PASS decisions, we fine-tune two LoRA-adapted classifiers on the generated labels. This step requires GPU infrastructure (we used 4x H100 GPUs) and is not reproducible in this notebook, but we describe the setup here for completeness.

**Two models are trained:**

| Model | Training data | What it estimates |
|-------|--------------|-------------------|
| Regular model | Actual product text | $P(Y \mid x)$ — how confident the model is *given* the text |
| Null model | Single space `" "` (empty input) | $P(Y \mid \emptyset)$ — the base rate, with *no* text |

The difference in their $\mathcal{F}$-entropies is the PVI.

**LoRA configuration** (applied to Llama-3.1-8B-Instruct):

```
Rank (r):        32        Target modules:  q, k, v, o, gate, up, down projections
Alpha:           64        Task:            Sequence classification (2 labels)
Dropout:         0.05      Precision:       bfloat16
```

**Training hyperparameters:**

```
Batch size: 12              Epochs: 3               Learning rate: 2e-4
Warmup ratio: 0.03          Weight decay: 0.01      Max sequence length: 1024
Gradient accumulation: 16   Gradient clipping: 1.0
```

The labeled dataset is split 80/10/10 into train/validation/test. The null model uses the same splits but with all text replaced by a single space — it learns only the label prior $P(Y)$.

See `src/finetuning.py` for the full implementation. The PVI results from our trained models are included in `data/` so you can reproduce the regression without re-training.

## 4. $\mathcal{F}$-Information Computation

After fine-tuning, we run both models on the held-out test set. For each listing, we get:
- **Regular model**: $P(Y \mid x)$ — the classifier's confidence given the actual text
- **Null model**: $P(Y \mid \emptyset)$ — the classifier's confidence given nothing

The $\mathcal{F}$-entropy for each listing is:

$$H_{\mathcal{F}}(Y \mid x) = -\log_2 P_{\text{model}}(y_i \mid x_i)$$

where $y_i$ is the gold label and $P_{\text{model}}$ is the fine-tuned classifier's predicted probability.

Here is the core computation from `v_info.py`:

In [ ]:
import numpy as np

def compute_pvi(prob_given_text: float, prob_given_empty: float) -> float:
    """Compute Pointwise V-Information for a single example.
    
    Args:
        prob_given_text: P(gold_label | product_text) from regular model
        prob_given_empty: P(gold_label | empty) from null model
    
    Returns:
        PVI in bits. Positive = text is informative.
    """
    H_yx = -np.log2(prob_given_text)    # entropy given text
    H_yb = -np.log2(prob_given_empty)   # entropy given nothing (baseline)
    return H_yb - H_yx

# Example: model is very confident given text, uncertain given nothing
pvi = compute_pvi(prob_given_text=0.95, prob_given_empty=0.55)
print(f"PVI = {pvi:.3f} bits")
print(f"  H(Y|text)  = {-np.log2(0.95):.3f} bits (low uncertainty — text is helpful)")
print(f"  H(Y|empty) = {-np.log2(0.55):.3f} bits (high uncertainty — no text)")

# Example: model is uncertain either way (uninformative text)
pvi2 = compute_pvi(prob_given_text=0.55, prob_given_empty=0.55)
print(f"\nPVI = {pvi2:.3f} bits (text provides no additional signal)")

## 5. Main Result

We include the PVI results from the main experiment (SELECT/PASS tokens, GPT-5-mini labels, balanced medium dataset). The data below is the **held-out test set** (10% split) from the class-balanced medium sample (~210k listings subsampled from 500k per period). Let's load them and run the paper's baseline regression:

$$\text{PVI}_i = \beta \cdot \text{after}_i + \alpha + \varepsilon_i$$

In [ ]:
import pandas as pd

before = pd.read_json("data/before_PVI.json", lines=True)
after = pd.read_json("data/after_PVI.json", lines=True)

print(f"Pre-ChatGPT listings:  {len(before):,}")
print(f"Post-ChatGPT listings: {len(after):,}")
print(f"\nColumns: {list(before.columns)}")
before.head(3)

In [ ]:
# Tag and combine
before["time_period"] = "before"
after["time_period"] = "after"
df = pd.concat([before, after], ignore_index=True)

print(f"Mean PVI (before): {df[df.time_period == 'before']['PVI'].mean():.4f}")
print(f"Mean PVI (after):  {df[df.time_period == 'after']['PVI'].mean():.4f}")
print(f"Difference:        {df[df.time_period == 'after']['PVI'].mean() - df[df.time_period == 'before']['PVI'].mean():.4f}")

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols('PVI ~ C(time_period, Treatment(reference="before"))', data=df).fit()
print(model.summary())

The coefficient on `after` is **0.143** (p < 0.01): post-ChatGPT listings contain 0.143 additional bits of machine-usable information on average.